<a href="https://colab.research.google.com/github/falcon23nk/collab-project/blob/main/pr%C3%A1ctica_2_1_sentimientos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Qué es el análisis de sentimiento**

Es una tarea de clasificación de texto. Dado un texto, el modelo decide si la intención utilizada es positiva, negativa o neutral.

No es "**entender**" el texto como lo haría una persona. Es detectar patrones de palabras y contexto que, estadísticamente, se asocian a cada categoría.

Para esta práctica en español, el modelo más adecuado es **RoBERTuito**. Es un modelo entrenado específicamente con **500 millones de tweets en español**, lo que le permite entender jerga, emojis, abreviaturas y texto informal.

Está diseñado para redes sociales, que es donde el análisis de sentimiento tiene más aplicación real. Y funciona bien en tareas de sentimiento, odio e ironía en español.

RoBERTuito no se entrenó desde cero. Es un modelo preentrenado: alguien invirtió muchísimo tiempo y recursos en entrenarlo con textos masivos para que "entienda" el español.

Luego ese modelo base se ajustó (mediante fine-tuning) con un conjunto más pequeño de tweets etiquetados como positivos, negativos o neutros. El resultado es un modelo especializado en análisis de sentimiento en español, listo para usar.

Esto es importante porque explica por qué tú no entrenas nada. Usas un modelo que ya existe, ya está entrenado, y solo lo llamas desde Colab. Es exactamente lo que haría un desarrollador profesional: **no reinventar la rueda, usar lo que ya funciona**.

# **Hugging Face**

Hugging Face es la plataforma donde viven estos modelos. Su función **pipeline** es una abstracción: tú le dices "quiero realizar un análisis de sentimiento" y "usa este modelo", y ella se encarga de todo lo demás (preprocesar el texto, pasarlo por el modelo, devolver el resultado).

Solo se necesita entender que pipeline es el "botón" que conecta el texto con un modelo ya entrenado.

El modelo devuelve dos cosas:

    Una etiqueta: POS, NEG o NEU

    Una confianza (score), un número entre 0 y 1

La confianza no es lo mismo que **"acierto"**. Es la seguridad del modelo en su propia predicción. Un score de 0.99 no garantiza que tenga razón; un score de 0.51 en una frase ambigua es normal y no es un fallo.

El modelo no lee intenciones, lee patrones. El guiño o el signo de interrogación alteran el contexto que el modelo aprendió durante el preentrenamiento, y eso mueve la predicción.

In [ ]:
from transformers import pipeline

clasificador = pipeline("sentiment-analysis", model="pysentimiento/robertuito-sentiment-analysis")
# carga un modelo de inteligencia artificial
# entrenado para analizar el sentimiento
# de textos en español, con especial enfoque
# en lenguaje informal y redes sociales

# === este es el texto  que se clasifica ===
texto = "Me encanta este módulo de IA"
# =====================================

resultado = clasificador(texto)
print(resultado)

In [ ]:
# clasifica un texto que sea negativo

texto = "Es pesimo ese trabajo"
resultado = clasificador(texto)
print(resultado)

In [ ]:
# clasifica un texto neutro
texto = "Es azul"
resultado = clasificador(texto)
print(resultado)

In [ ]:
# clasifica un texto irónico
texto = "Tiene menos luces que una farola"
resultado = clasificador(texto)
print(resultado)

In [ ]:
# clasifica un texto usando emojis
texto = "😡"
resultado = clasificador(texto)
print(resultado)

**¿Cómo funciona el análisis de sentimientos? ¿Qué límites tiene?**

*Escribe aquí tu respuesta.*

# **Práctica**

Implementa en python un analizador de comentarios. Los comentarios positivos y nuetros serán aceptados, salvo que contengan alguna palabra de la lista negra de palabras. Los comentarios negativos que contengan alguna palabra de la lista negra o que tengan un score de más de 0.5 serán rechazadas.

In [4]:
import re
from transformers import pipeline

clasificador = pipeline("sentiment-analysis", model="pysentimiento/robertuito-sentiment-analysis")

lista_negra = [
    "pendejo", "mierda", "estafa", "basura", "horrible",
    "pestilente", "asco", "inútil", "pésimo", "fraude"
]

frases_prueba = [
    "Ese vendedor es un pendejo y no responde los mensajes.",
    "El servicio me pareció muy rápido y atento.",
    "Siento que este servicio fue una estafa completa.",
    "El producto llegó en buen estado a mi casa.",
    "La aplicación es una mierda, se cierra sola a cada rato.",
    "No estoy muy seguro de si me convence esta opción.",
    "Todo el contenido del curso me pareció pésimo de principio a fin.",
    "La clase estuvo bastante clara e instructiva.",
    "El diseño nuevo me parece una basura total.",
    "El soporte técnico tardó un poco en responder."
]

def borrar_palabras_prohibidas(texto):
    texto_limpio = texto
    encontrada = False
    for palabra in lista_negra:
        if palabra.lower() in texto_limpio.lower():
            encontrada = True
            texto_limpio = re.sub(re.escape(palabra), "***", texto_limpio, flags=re.IGNORECASE)
    return texto_limpio, encontrada

def analizar_comentario(comentario):
    if not comentario.strip():
        return "RECHAZADO (Comentario vacío)"

    texto_limpio, tiene_palabra_prohibida = borrar_palabras_prohibidas(comentario)
    prediccion = clasificador(comentario)[0]
    etiqueta = prediccion['label']

    # Conversión del score de 0.0 - 1.0 a una valoración de 0 a 100
    score_100 = round(prediccion['score'] * 100)

    if tiene_palabra_prohibida or (etiqueta == "NEG" and score_100 > 50):
        estado = "RECHAZADO"
    else:
        estado = "ACEPTADO"

    return f"Original: \"{comentario}\"\nLimpio:   \"{texto_limpio}\"\nResultado: {estado} [{etiqueta} - {score_100}/100]\n"

for frase in frases_prueba:
    print(analizar_comentario(frase))
    print("-" * 50)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Original: "Ese vendedor es un pendejo y no responde los mensajes."
Limpio:   "Ese vendedor es un *** y no responde los mensajes."
Resultado: RECHAZADO [NEG - 98/100]

--------------------------------------------------
Original: "El servicio me pareció muy rápido y atento."
Limpio:   "El servicio me pareció muy rápido y atento."
Resultado: ACEPTADO [POS - 94/100]

--------------------------------------------------
Original: "Siento que este servicio fue una estafa completa."
Limpio:   "Siento que este servicio fue una *** completa."
Resultado: RECHAZADO [NEG - 98/100]

--------------------------------------------------
Original: "El producto llegó en buen estado a mi casa."
Limpio:   "El producto llegó en buen estado a mi casa."
Resultado: ACEPTADO [POS - 89/100]

--------------------------------------------------
Original: "La aplicación es una mierda, se cierra sola a cada rato."
Limpio:   "La aplicación es una ***, se cierra sola a cada rato."
Resultado: RECHAZADO [NEG - 97/100]

---